In [ ]:
import sys

sys.path.append("/home/alberto/UnReflectAnything/")
import torch
from rich import print

from dataset.rgbp import *
from utilities.visualization import panelize, rgb

%load_ext autoreload
%autoreload 2    

In [ ]:
from polar_highlighter import PolarHighlighter, get_soft_highlight_map
add_polar_highlights = PolarHighlighter(height=448, width=448).cuda()
from main import load_and_process_config
from dataset import from_config
from utilities import tensor_dict_summarize
from utilities.visualization import highlights_rerun_show
config = load_and_process_config("config_train.yaml")
dataset = HOUSECAT6D_Dataset(
    # scene_names=["scene01"],
    target_size=[448, 448],
    resize_mode="resize",
    rho_s=0.6,
    eps=1e-8,
)

In [ ]:
config = load_and_process_config("config_train.yaml")
config.BATCH_SIZE = 1
dataloader = torch.utils.data.DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)
idataloade = iter(dataloader)

# Test loading a batch  
for b, batch in enumerate(dataloader):
    break

diffuse = batch["diffuse"].cuda()
# pol = torch.cat([batch["S0"], batch["S1"], batch["S2"]], dim=1).cuda()
random_light_pos = add_polar_highlights.sample_light_source(
    dist_to_camera=config.LIGHT_DISTANCE_RANGE,
    left_right_angle=config.LIGHT_LEFT_RIGHT_ANGLE,
    above_below_angle=config.LIGHT_ABOVE_BELOW_ANGLE,
    batch_size=config.BATCH_SIZE,
    device="cuda",
)
highlight_result = add_polar_highlights(
    rgb=batch["diffuse"].to("cuda", non_blocking=True),
    light_pos=random_light_pos,
    noise=config.NOISE,
    noise_type=config.NOISE_TYPE, 
    noise_octaves=config.NOISE_OCTAVES,
    noise_persistence=config.NOISE_PERSISTENCE,
    surface_roughness=config.SURFACE_ROUGHNESS,
    intensity=config.INTENSITY,
)

# Test the function
soft_highlight_map = get_soft_highlight_map(diffuse, threshold=0.8)
combined_highlights = highlight_result["highlight"] + soft_highlight_map

# Use the weighted combination for better control
highlight_result["highlight_both"] = combined_highlights.clamp(0,1)

rgb(
    panelize(
        rgb(diffuse, resize=(448, 448), as_tensor=True),
        rgb(soft_highlight_map[0, 0], resize=(448, 448), as_tensor=True,colormap="gray"),
        rgb(highlight_result["highlight"][0], resize=(448, 448), colormap="jet", as_tensor=True),
        rgb(highlight_result["highlight_both"][0], resize=(448, 448), colormap="gray", as_tensor=True),
        rgb(highlight_result["rgb_highlighted"][0], resize=(448, 448), as_tensor=True),
    )
)
# highlights_rerun_show(highlight_result, batch,)

In [ ]:
def highlights_rerun_show(
    highlight_result, 
    batch, 
    resolution=(448, 448), 
    width=1700, 
    height=800,
    show_normals=True, 
    show_light_dir=True, 
    show_view_dir=True
):
    """
    Visualize highlight and geometry info with rerun.
    
    Args:
        highlight_result (dict): Outputs from PolarHighlighter.
        batch (dict): Associated batch. Uses batch["diffuse"] for colors.
        resolution (tuple): Output resolution (height, width) for camera.
        width (int): Width of rerun viewer.
        height (int): Height of rerun viewer.
        show_normals (bool): Whether to plot surface normals as 3D arrows.
        show_light_dir (bool): Whether to plot light direction arrows.
        show_view_dir (bool): Whether to plot view direction arrows.
    """
    import rerun as rr
    import torch

    rr.init("polar_highlighter")
    rr.log("/", rr.ViewCoordinates.RDF)
    rr.log(
        "/cam",
        rr.Pinhole(
            image_from_camera=highlight_result["intrinsic"][0].cpu().numpy(),  # (3, 3)
            resolution=resolution,  # (2,)
            camera_xyz=rr.components.ViewCoordinates.RDF,  # Default: X=Right, Y=Down, Z=Forward
        ),
    )
    # Light position and color (white)
    rr.log(
        "/light",
        rr.Points3D(
            positions=highlight_result["light_pos"].cpu().numpy().reshape(1, 3),
            colors=torch.tensor([1.0, 1.0, 1.0]).cpu().numpy().reshape(1, 3),
            radii=0.05,
        ),
    )
    # Point cloud of (x, y, z) points colored by diffuse RGB
    rr.log(
        "/pcloud",
        rr.Points3D(
            positions=highlight_result["pcloud"].view(1, 3, -1)[0].permute(1, 0).cpu().numpy(),
            colors=batch["diffuse"].view(1, 3, -1)[0].permute(1, 0).cpu().numpy(),
        ),
    )
    # Constants for visualization clarity
    CLOUD_DOWNS = 10
    ARROW_LENGTH = 0.1
    ARROW_RADIUS = 0.005

    cloud = highlight_result["pcloud"][:, :, ::CLOUD_DOWNS, ::CLOUD_DOWNS]

    if show_normals:
        # Normals as arrows, colored using HSV-like mapping for direction
        normals = highlight_result["normals"][:, :, ::CLOUD_DOWNS, ::CLOUD_DOWNS]
        rr.log(
            "/normals",
            rr.Arrows3D(
                origins=cloud.reshape(1, 3, -1)[0].permute(1, 0).cpu().numpy(),
                vectors=-normals.reshape(1, 3, -1)[0].permute(1, 0).cpu().numpy() * ARROW_LENGTH,
                colors=torch.nn.functional.normalize(normals.reshape(1, 3, -1)[0].permute(1, 0), dim=1)
                .cpu()
                .numpy()
                * 0.5
                + 0.5,  # HSV-like coloring
                radii=ARROW_RADIUS,
            ),
        )
    if show_light_dir:
        # Light direction arrows (white)
        light_dir = highlight_result["light_dir"][:, :, ::CLOUD_DOWNS, ::CLOUD_DOWNS]
        rr.log(
            "/light_dir",
            rr.Arrows3D(
                origins=cloud.reshape(1, 3, -1)[0].permute(1, 0).cpu().numpy(),
                vectors=light_dir.reshape(1, 3, -1)[0].permute(1, 0).cpu().numpy() * ARROW_LENGTH,
                colors=torch.ones_like(light_dir.reshape(1, 3, -1)[0].permute(1, 0)).cpu().numpy(),
                radii=ARROW_RADIUS,
            ),
        )
    if show_view_dir:
        # View direction arrows (red)
        view_dir = highlight_result["view_dir"][:, :, ::CLOUD_DOWNS, ::CLOUD_DOWNS]
        rr.log(
            "/view_dir",
            rr.Arrows3D(
                origins=cloud.reshape(1, 3, -1)[0].permute(1, 0).cpu().numpy(),
                vectors=-view_dir.reshape(1, 3, -1)[0].permute(1, 0).cpu().numpy() * ARROW_LENGTH,
                colors=torch.tensor([1.0, 0.0, 0.0])
                .expand_as(view_dir.reshape(1, 3, -1)[0].permute(1, 0))
                .cpu()
                .numpy(),
                radii=ARROW_RADIUS,
            ),
        )
    rr.notebook_show(width=width, height=height)


In [ ]:
from dataset.highlight import HighlightDataset
from dataset.mono3d_dataset import SCARED
import os
purescared = HighlightDataset(
    SCARED(
        # There are 10 frames of SCARED in the dataset/sample_datasets/SCARED/ directory
        path=os.path.expandvars("$DATASET_DIR/SCARED/"),
        vids=SCARED.videonames(),
    ),
    brightness_threshold=0.9,
    rect_size=(384, 384),
    return_mask=True,
    return_rect=True,
)

batch= purescared[0]


In [ ]:
448*448

In [ ]:
rgb(is_highlight.int(),colormap="gray")

In [ ]:
loss = UnReflectLoss(
).cuda()

input_data = {
    "diffuse": batch["rgb"].cuda(),
    "rgb": result_ph["rgb_highlighted"].cuda(),
    "specular": batch["specular"].cuda(),
    "highlight": result_ph["highlight"].cuda()#.repeat(1, 4, 1, 1),
    }
decomposition = model(input_data)
# decomposition["diffuse"] = input_data["diffuse"].clone()
# decomposition["highlight"] = input_data["highlight"].clone()

lossv = loss(decomposition, input_data)
print("Decomposition")
print([{k: (v.shape,v.min().item(),v.max().item()) for k, v in decomposition.items()}])

print("Input data")
print([{k: (v.shape,v.min().item(),v.max().item()) for k, v in input_data.items()}])
print("Loss")
print([{k: v for k, v in lossv.items()}])


In [ ]:
rgb()

In [ ]:
import matplotlib.pyplot as plt

plt.subplot(1, 3, 1)
plt.imshow(
    torch.cat(
        [
            input_data["rgb"][:, 0],
            torch.zeros_like(input_data["rgb"][:, 0]),
            torch.zeros_like(input_data["rgb"][:, 0]),
        ]
    )
    .permute(1, 2, 0)
    .detach()
    .cpu()
)
torch.colo
plt.subplot(1, 3, 2)
plt.imshow(
    torch.cat(
        [
            torch.zeros_like(input_data["rgb"][:, 0]),
            input_data["rgb"][:, 1],
            torch.zeros_like(input_data["rgb"][:, 0]),
        ]
    )
    .permute(1, 2, 0)
    .detach()
    .cpu()
)
torch.colo
plt.subplot(1, 3, 3)
plt.imshow(
    torch.cat(
        [
            torch.zeros_like(input_data["rgb"][:, 2]),
            torch.zeros_like(input_data["rgb"][:, 0]),
            input_data["rgb"][:, 1],
        ]
    )
    .permute(1, 2, 0)
    .detach()
    .cpu()
)
torch.colo
plt.show()


In [ ]:
rgb(panelize(
    rgb(input_data["diffuse"][0].detach(), resize=(224, 224), as_tensor=True),
    rgb(input_data["highlight"][0].detach(), resize=(224, 224), as_tensor=True,colormap="gray"),
    rgb(input_data["rgb"][0].detach(), resize=(224, 224), as_tensor=True),
))
rgb(panelize(
    rgb(decomposition["diffuse"][0].detach(), resize=(224, 224), as_tensor=True),
    rgb(decomposition["highlight"][0].detach(), resize=(224, 224), as_tensor=True,colormap="gray"),
))

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(input_data["rgb"][0].mean(dim=0).detach().cpu()
        )
plt.colorbar()
plt.show()

In [ ]:
out = distill(batch)
print(out["specular"].shape)
print(out["diffuse"].shape)
print(out["highlight"].shape)
print([l.shape for l in out["rgb_tokens"]])

In [ ]:
out = model(batch)
print(out["specular"].shape)
print(out["diffuse"].shape)
print(out["highlight"].shape)
print([l.shape for l in out["rgb_tokens"]])
print([l.shape for l in out["pol_tokens"]])
print([l.shape for l in out["cross_tokens"]])

In [ ]:
model.dinov3.config

In [ ]:
rgb(
    panelize(
        rgb(
            batch["rgb"][0],
            as_tensor=True,
            resize=(892, 1140),
            colormap="gray",
        ),
        panelize(
            panelize(
                rgb(
                    0.2 * batch["specular"][0] + 0.8 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
                rgb(
                    0.4 * batch["specular"][0] + 0.6 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
                rgb(
                    0.6 * batch["specular"][0] + 0.4 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
                rgb(
                    0.8 * batch["specular"][0] + 0.2 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
            ),
            panelize(
                rgb(
                    0.8 * batch["specular"][0] + 0.2 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
                rgb(
                    0.6 * batch["specular"][0] + 0.4 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
                rgb(
                    0.4 * batch["specular"][0] + 0.6 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
                rgb(
                    0.2 * batch["specular"][0] + 0.8 * batch["diffuse"][0],
                    as_tensor=True,
                    resize=(892 // 2, 1140 // 2),
                ),
            ),
            mode="vertical",
        ),
        resize_to_match=False,
    )
)

In [ ]:
import yaml
from dotmap import DotMap

from projections import ReflectionWarp

# Initialize the module
height, width = batch["rgb"].shape[2:]
reflection_warp = ReflectionWarp(height, width)
reflection_warp = reflection_warp.cuda()  # Move to GPU


light_color = torch.tensor([1.0, 0.8, 0.8]).cuda()  # Warm light

from pipelines.depth.depth import DepthPipeline

CONFIG_PATH = "../config.yaml"
with open(CONFIG_PATH, "r") as f:
    config_yaml = yaml.safe_load(f)
    config_parameters = config_yaml["parameters"]
    config_training_dict = {
        k: v.get("value") for k, v in config_parameters.items() if v is not None
    }
    config = DotMap(config_training_dict)
    config.IMAGE_HEIGHT = height
    config.IMAGE_WIDTH = width
depthPipeline = DepthPipeline(config, model="", device="cuda")

light_position = torch.randn((1, 3)) * config.DEPTH_SCALE_FACTOR / 2
light_position[0, 1:] = -torch.abs(light_position[0, 1:])
torch.cuda.empty_cache()
with torch.no_grad():
    depth_map = depthPipeline.depth(batch["rgb"].cuda())

# Call with point light
result = reflection_warp.forward_point_light(
    source_image=batch["rgb"][0:1].cuda(),
    depth_map=depth_map[0:1].cuda(),
    camera_intrinsics=batch["intrinsics"].cuda()[0:1],
    camera_pose=torch.eye(4)
    .unsqueeze(0)
    .repeat(batch["rgb"].shape[0], 1, 1)
    .cuda()[0:1],
    light_position=light_position.cuda(),
    light_intensity=100.0,
    light_color=light_color.cuda(),
    surface_roughness=0.1,  # Slightly rough surface
    reflection_strength=0.9,  # Strong reflections
    return_mask=True,
    return_artifacts=True,
)
rgb(
    panelize(
        rgb(result["warped"][0], as_tensor=True),
        rgb(result["reflection_only"][0], colormap="gray", as_tensor=True),
        rgb(batch["rgb"][0], as_tensor=True),
    )
)

In [ ]:
result.keys()

In [ ]:
(
    batch["rgb"].shape,
    depth_map.shape,
    batch["intrinsics"].shape,
    torch.eye(4).unsqueeze(0).repeat(batch["rgb"].shape[0], 1, 1).cuda().shape,
)

In [ ]:
result = reflection_warp.forward_point_light(
    source_image=batch["rgb"][0:1].cuda(),
    depth_map=depth_map[0:1].cuda(),
    camera_intrinsics=batch["intrinsics"].cuda()[0:1],
    camera_pose=torch.eye(4)
    .unsqueeze(0)
    .repeat(batch["rgb"].shape[0], 1, 1)
    .cuda()[0:1],
    light_position=light_position.cuda(),
    light_intensity=100.0,
    light_color=light_color.cuda(),
    surface_roughness=0.1,  # Slightly rough surface
    reflection_strength=0.9,  # Strong reflections
    return_mask=True,
    return_artifacts=True,
)

In [ ]:
rgb(batch["rgb"][0])
rgb(depth_map[0], colormap="jet")